# Snowflake AISQL Functions — Health Demo

This notebook demonstrates **Snowflake Cortex AI Functions** (AISQL) using synthetic health data created inline.

Functions covered:
- `AI_CLASSIFY` — Categorise clinical notes
- `AI_EXTRACT` — Extract structured data from free text
- `AI_SENTIMENT` — Gauge patient feedback sentiment
- `AI_FILTER` — Boolean filtering with natural language
- `AI_COMPLETE` — Generative responses for clinical Q&A
- `AI_TRANSLATE` — Translate health information
- `AI_REDACT` — Remove PII from clinical text
- `AI_AGG` — Aggregate insights across multiple rows
- `SNOWFLAKE.CORTEX.SUMMARIZE` — Summarise lengthy text

## 1. Create Sample Health Data

In [ ]:
%%sql -r dataframe_1
USE DATABASE HACKATHON_DB;
USE SCHEMA DATA;

In [ ]:
%%sql -r create_health_notes
CREATE OR REPLACE TEMPORARY TABLE health_notes AS
SELECT * FROM VALUES
  (1, 'Patient presented with persistent cough lasting 3 weeks, mild fever 37.8°C. Chest X-ray ordered. Suspect lower respiratory tract infection. Started on amoxicillin 500mg TDS.', 'GP Consultation', 'Dr Sarah Thompson'),
  (2, 'John Smith, NHI ZZZ1234, DOB 15/03/1985. Diagnosed with Type 2 Diabetes Mellitus. HbA1c 8.2%. Commenced metformin 500mg BD. Referred to dietitian. Phone: 021-555-0147.', 'Chronic Disease', 'Dr Aroha Williams'),
  (3, 'Follow-up post knee arthroscopy. Patient reports improved mobility, pain 3/10. Wound healing well. Physio exercises progressing. Return in 6 weeks.', 'Surgical Follow-up', 'Mr James Chen'),
  (4, 'Mental health review: Patient experiencing increased anxiety and low mood following job loss. PHQ-9 score 14 (moderate depression). Discussed CBT options and social supports. Safety plan reviewed — no current SI.', 'Mental Health', 'Dr Mere Patel'),
  (5, 'Child aged 4 years, brought in by mother. High fever 39.5°C for 2 days, refusing food, pulling at right ear. Otoscopy: bulging erythematous TM. Diagnosis: Acute otitis media. Prescribed amoxicillin 250mg TDS x 5 days.', 'Paediatrics', 'Dr Liam OConnor'),
  (6, 'Mary Johnson, NHI ABC5678, 72yo female. Presented to ED with sudden onset left-sided weakness and slurred speech at 14:30. CT head: no haemorrhage. Thrombolysis initiated within window. Admitted to stroke unit.', 'Emergency', 'Dr Raj Patel')
AS t(note_id, clinical_note, category, clinician);

SELECT * FROM health_notes;

## 2. Create Patient Feedback Data

In [ ]:
%%sql -r create_feedback
CREATE OR REPLACE TEMPORARY TABLE patient_feedback AS
SELECT * FROM VALUES
  (1, 'The doctor was very thorough and explained everything clearly. I felt listened to and the treatment plan makes sense.'),
  (2, 'Waited over 2 hours past my appointment time. Receptionist was rude. The doctor was fine but the overall experience was frustrating.'),
  (3, 'Excellent care from the nursing team. They went above and beyond to make me comfortable after my procedure.'),
  (4, 'I am worried the medication is not working. Side effects are bad and nobody follows up with me. Feeling quite hopeless.'),
  (5, 'Average visit. Got what I needed but felt rushed. Would have appreciated more time to ask questions.'),
  (6, 'The telehealth appointment was convenient but the connection kept dropping. Hard to discuss sensitive issues over video.')
AS t(feedback_id, feedback_text);

SELECT * FROM patient_feedback;

---
## 3. AI_CLASSIFY — Categorise Clinical Notes

Automatically classify free-text clinical notes into clinical specialties.

In [ ]:
%%sql -r classify_demo
SELECT
    note_id,
    LEFT(clinical_note, 60) || '...' AS note_preview,
    category AS actual_category,
    AI_CLASSIFY(
        clinical_note,
        ['Respiratory', 'Endocrinology', 'Orthopaedics', 'Mental Health', 'Paediatrics', 'Neurology/Stroke', 'Emergency']
    ):labels[0]::STRING AS ai_category
FROM health_notes;

---
## 4. AI_EXTRACT — Extract Structured Data from Clinical Text

Pull out key clinical entities like medications, dosages, and diagnoses.

In [ ]:
%%sql -r extract_demo
SELECT
    note_id,
    clinician,
    AI_EXTRACT(clinical_note, ['medication', 'dosage', 'diagnosis', 'temperature'])::VARIANT AS extracted
FROM health_notes;

---
## 5. AI_SENTIMENT — Patient Feedback Analysis

Score patient feedback on a scale from -1 (very negative) to +1 (very positive).

In [ ]:
%%sql -r sentiment_demo
SELECT
    feedback_id,
    LEFT(feedback_text, 80) || '...' AS feedback_preview,
    AI_SENTIMENT(feedback_text):categories[0]:sentiment::STRING AS sentiment_score,
    CASE
        WHEN AI_SENTIMENT(feedback_text):categories[0]:sentiment::STRING = 'positive' THEN 'Positive'
        WHEN AI_SENTIMENT(feedback_text):categories[0]:sentiment::STRING = 'negative' THEN 'Negative'
        WHEN AI_SENTIMENT(feedback_text):categories[0]:sentiment::STRING = 'mixed' THEN 'Mixed'
        ELSE 'Neutral'
    END AS sentiment_label
FROM patient_feedback
ORDER BY sentiment_label;

---
## 6. AI_FILTER — Natural Language Boolean Filtering

Filter rows using plain English questions instead of complex WHERE clauses.

In [ ]:
%%sql -r filter_demo
SELECT
    note_id,
    clinician,
    LEFT(clinical_note, 80) || '...' AS note_preview
FROM health_notes
WHERE AI_FILTER(PROMPT('Does this note involve prescribing antibiotics? {0}', clinical_note));

---
## 7. AI_COMPLETE — Clinical Q&A Generation

Use an LLM to generate clinical guidance or answer questions given context.

In [ ]:
%%sql -r complete_demo
SELECT
    note_id,
    clinician,
    AI_COMPLETE(
        'mistral-large2',
        'Based on the following clinical note, suggest appropriate follow-up actions in 2-3 bullet points:\n\n' || clinical_note
    ) AS suggested_followup
FROM health_notes
WHERE note_id IN (1, 4, 6);

---
## 8. AI_TRANSLATE — Translate Health Information

Translate patient-facing health advice into Te Reo Māori and other languages.

In [ ]:
%%sql -r translate_demo
SELECT
    original_text,
    AI_TRANSLATE(original_text, 'en', 'de') AS german,
    AI_TRANSLATE(original_text, 'en', 'zh') AS chinese,
    AI_TRANSLATE(original_text, 'en', 'fr') AS french
FROM (SELECT 'Please take your medication twice daily with food. If you experience any side effects such as nausea or dizziness, contact your doctor immediately.' AS original_text);

---
## 9. AI_REDACT — Remove PII from Clinical Notes

Automatically redact personally identifiable information (names, NHI numbers, DOBs, phone numbers).

In [ ]:
%%sql -r redact_demo
SELECT
    note_id,
    clinical_note AS original_note,
    AI_REDACT(clinical_note) AS redacted_note
FROM health_notes
WHERE note_id IN (2, 6);

---
## 10. AI_AGG — Aggregate Insights Across Multiple Records

Summarise themes and patterns across all patient feedback in a single call.

In [ ]:
%%sql -r agg_demo
SELECT AI_AGG(
    feedback_text,
    'Identify the top 3 themes from this patient feedback. For each theme provide: the theme name, how many comments relate to it, and a recommendation for the health service.'
) AS aggregated_insights
FROM patient_feedback;

---
## 11. SNOWFLAKE.CORTEX.SUMMARIZE — Summarise Lengthy Clinical Text

Condense long clinical notes into concise summaries for handover or referral.

In [ ]:
%%sql -r summarize_demo
SELECT
    note_id,
    clinician,
    SNOWFLAKE.CORTEX.SUMMARIZE(clinical_note) AS summary
FROM health_notes;

---
## 12. Combining Functions — Triage Pipeline

Combine multiple AISQL functions to build an automated clinical triage pipeline.

In [ ]:
%%sql -r pipeline_demo
SELECT
    note_id,
    clinician,
    AI_CLASSIFY(
        clinical_note,
        ['Urgent', 'Routine', 'Preventive'],
        {'task_description': 'Classify the clinical note by urgency: Urgent requires immediate attention, Routine is standard follow-up, Preventive is wellness or screening'}
    ):labels[0]::STRING AS urgency,
    AI_EXTRACT(clinical_note, ['diagnosis', 'medication'])::VARIANT AS key_entities,
    AI_FILTER(PROMPT('Does this note mention prescribing medication to the patient? {0}', clinical_note)) AS involves_medication,
    SNOWFLAKE.CORTEX.SUMMARIZE(clinical_note) AS one_line_summary
FROM health_notes;

---

## Summary

| Function | Use Case |
|----------|----------|
| `AI_CLASSIFY` | Auto-categorise notes into specialties or urgency levels |
| `AI_EXTRACT` | Pull medications, diagnoses, vitals from free text |
| `AI_SENTIMENT` | Score patient satisfaction from feedback |
| `AI_FILTER` | Natural language WHERE clauses |
| `AI_COMPLETE` | Generate follow-up suggestions, discharge summaries |
| `AI_TRANSLATE` | Multi-language patient communications |
| `AI_REDACT` | PII removal for de-identification |
| `AI_AGG` | Aggregate insights across many records |
| `SUMMARIZE` | Condense long notes for handover |

All functions run **directly in SQL** — no external APIs, no data leaving Snowflake.